<a href="https://colab.research.google.com/github/isumakm/Weather-Prediction-and-Crop-Recommendation-System-/blob/Single-Crop-Analysis/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load data (if starting fresh)
df = pd.read_csv('Crop_training_data_FULL _1.csv')

1. Prepare features and target

In [ ]:
# Drop 'texture' (redundant) and keep 'texture_code' (ordinal)
X = df.drop(columns=['suitability', 'suitability_class', 'texture'])
y = df['suitability_class'].map({'Suitable': 1, 'Unsuitable': 0})  # binary encoding

# Verify
print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (2100, 11)
Target distribution:
 suitability_class
0    1246
1     854
Name: count, dtype: int64


2. Identify column types

In [ ]:
numeric_features = ['temperature', 'rainfall', 'sunshine_hours', 'ph',
                    'organic_carbon', 'cec', 'awc', 'bulk_density',
                    'rooting_depth_m', 'texture_code']  # texture_code is numeric

categorical_features = ['crop']  # one-hot encode

3. Create preprocessing transformers

In [ ]:
numeric_transformer = StandardScaler()  # scales to mean=0, std=1
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

4. Split data (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (1680, 11), Test size: (420, 11)


5. Fit preprocessor on training data and transform both sets

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# The result is a dense numpy array (if all transformers output dense)
print("Processed train shape:", X_train_processed.shape)

Processed train shape: (1680, 31)


6. (Optional) Save preprocessor for later use

In [ ]:
import os
import joblib

# Create the directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Now save
joblib.dump(preprocessor, 'models/preprocessor.pkl')

['models/preprocessor.pkl']